# Chapter 7 - Lab 1: <font color='blue'>Sequential Pipeline with LlamaIndex AgentWorkflow</font>

**<font color='purple'>Goal</font>**:
Build a 4-agent **sequential pipeline** for financial health analysis using LlamaIndex's `AgentWorkflow`. The four agents — Fundamental, Profitability, Liquidity, Supervisor — pass control through **explicit handoffs** and share data via a `Context` object.

This is the *primary* implementation in this chapter; Labs 2 and 3 solve the same task with the OpenAI Agents SDK and AutoGen so you can compare three architectural styles head-to-head.

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex** — `FunctionAgent`, `AgentWorkflow`, `Context` for shared state.
* **OpenAI** `gpt-4o` — strong reasoning needed for the supervisor's synthesis.
* **Stub data** — bundled with the chapter so labs run without a Financial Modeling Prep key.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally — assume the env var is already set.
    pass

## 3. Bootstrap the shared models and helpers

Chapter 7 labs share a `common.py` module that defines the Pydantic models and helpers used across the chapter. If you have cloned the book's repository, `common.py` is already on disk; otherwise the cell below downloads it for you.

In [ ]:
import os, urllib.request

if not os.path.exists('common.py'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/PacktPublishing/Building-AI-Agents-for-Finance-/main/Chapter%207/common.py',
        'common.py',
    )

from common import (
    PROFITABILITY_THRESHOLDS, LIQUIDITY_THRESHOLDS,
    format_threshold_prompt, stub_lookup,
)

print('Thresholds and stub data loaded.')

## 4. Define the four agent tools

Each agent in the pipeline has a dedicated tool that reads from and writes to a shared `Context`. This is the **shared-state communication pattern** (§4.1 of the chapter): instead of passing data through return values, agents collaborate via a typed state object.

In [ ]:
from llama_index.core.workflow import Context

async def get_fundamentals(ctx: Context, ticker: str) -> str:
    """Used by FundamentalAgent. Reads stub data into shared state."""
    data = stub_lookup(ticker)
    state = await ctx.get('state')
    if state['ticker'] == '':
        state['ticker'] = ticker
    state['ratios'] = {
        **data['profitability'], **data['liquidity'],
    }
    await ctx.set('state', state)
    return f'Ratios extracted for {ticker}.'


async def get_profitability_ratios(ctx: Context) -> str:
    """Used by ProfitabilityAgent. Scores against thresholds."""
    state = await ctx.get('state')
    if not state['ratios']:
        return 'No ratios — run get_fundamentals first.'
    state['profitability_ratios'] = {
        k: state['ratios'][k] for k in PROFITABILITY_THRESHOLDS
    }
    state['threshold_profitability_comments'] = format_threshold_prompt(
        PROFITABILITY_THRESHOLDS, 'Profitability',
    )
    await ctx.set('state', state)
    return f'Profitability scored for {state["ticker"]}.'


async def get_liquidity_ratios(ctx: Context) -> str:
    """Used by LiquidityAgent. Scores against thresholds."""
    state = await ctx.get('state')
    if not state['ratios']:
        return 'No ratios — run get_fundamentals first.'
    state['liquidity_ratios'] = {
        k: state['ratios'][k] for k in LIQUIDITY_THRESHOLDS
    }
    state['threshold_liquidity_comments'] = format_threshold_prompt(
        LIQUIDITY_THRESHOLDS, 'Liquidity',
    )
    await ctx.set('state', state)
    return f'Liquidity scored for {state["ticker"]}.'


async def get_overall_comments(ctx: Context) -> str:
    """Used by SupervisorAgent. Produces the final health verdict."""
    state = await ctx.get('state')
    if state['threshold_profitability_comments'] is None:
        return 'Profitability scoring not done.'
    if state['threshold_liquidity_comments'] is None:
        return 'Liquidity scoring not done.'
    state['overall_comments'] = 'Synthesis complete.'
    await ctx.set('state', state)
    return 'Overall comments produced.'

## 5. Define the four agents and their handoffs

Each agent has a focused system prompt, its dedicated tool, and an explicit `can_handoff_to` declaration that names the *next* agent in the pipeline. This is the explicit-next-agent handoff pattern (§4.2). The Supervisor closes the loop by being able to hand back to Fundamental if a re-run is needed.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)

fundamental_agent = FunctionAgent(
    name='FundamentalAgent',
    description='Collect fundamental ratios for a ticker.',
    system_prompt=(
        'You are the fundamental analyst. Extract ratios for the ticker. '
        'Once extracted, hand off to the ProfitabilityAgent.'
    ),
    llm=llm,
    tools=[get_fundamentals],
    can_handoff_to=['ProfitabilityAgent'],
)

profitability_agent = FunctionAgent(
    name='ProfitabilityAgent',
    description='Score profitability ratios against thresholds.',
    system_prompt=(
        'You are the ProfitabilityAgent. Score the profitability ratios against '
        'threshold_profitability_comments. Then hand off to the LiquidityAgent.'
    ),
    llm=llm,
    tools=[get_profitability_ratios],
    can_handoff_to=['LiquidityAgent'],
)

liquidity_agent = FunctionAgent(
    name='LiquidityAgent',
    description='Score liquidity ratios against thresholds.',
    system_prompt=(
        'You are the LiquidityAgent. Score the liquidity ratios. '
        'Then hand off to the SupervisorAgent.'
    ),
    llm=llm,
    tools=[get_liquidity_ratios],
    can_handoff_to=['SupervisorAgent'],
)

supervisor_agent = FunctionAgent(
    name='SupervisorAgent',
    description='Synthesise overall financial health verdict.',
    system_prompt=(
        'You are the Supervisor. Combine profitability and liquidity assessments '
        'into a final verdict on the company. Justify with data.'
    ),
    llm=llm,
    tools=[get_overall_comments],
    can_handoff_to=['FundamentalAgent'],
)

## 6. Assemble the workflow

`AgentWorkflow` wires the agents together with the initial shared state. We name `FundamentalAgent` as the **root** — the first agent to run when the workflow starts.

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow

agent_workflow = AgentWorkflow(
    agents=[fundamental_agent, profitability_agent,
            liquidity_agent, supervisor_agent],
    root_agent=fundamental_agent.name,
    initial_state={
        'ratios': {},
        'ticker': '',
        'profitability_ratios': {},
        'liquidity_ratios': {},
        'threshold_profitability_comments': None,
        'threshold_liquidity_comments': None,
        'overall_comments': None,
    },
)

## 7. Run the workflow and observe agent transitions

Stream events from the workflow handler. Whenever the active agent changes, print a banner — this is your live view of the pipeline making handoffs.

In [ ]:
ticker = 'AAPL'
handler = agent_workflow.run(
    user_msg=f'Provide the fundamental analysis of {ticker} and comment on the financial health.'
)

async for event in handler.stream_events():
    if hasattr(event, 'current_agent_name'):
        print(f'\n{"=" * 40}\n  Active agent: {event.current_agent_name}\n{"=" * 40}')
    if hasattr(event, 'delta') and event.delta:
        print(event.delta, end='', flush=True)

result = await handler
print('\n\nFinal result:\n' + str(result))

## 8. Results

You watched four specialist agents pass control through a fixed sequence, sharing data via a typed `Context` object. **What to notice about the sequential-pipeline style:**

* **Predictability.** The handoff graph is declared up-front — you know exactly which agents   will run, in what order. Easy to audit, easy to reason about.
* **Specialisation.** Each agent has a narrow system prompt focused on one job. Quality   generally improves over a generalist agent.
* **Shared state vs message passing.** Here we use shared state. Compare with Lab 2 where the   OpenAI SDK uses handoff messages, and Lab 3 where AutoGen uses raw conversation.
* **Trade-off.** Sequential pipelines do not parallelise. Lab 5 adds an outer parallel layer   to analyse a whole portfolio concurrently.